# Microplastic Detection in Soil Samples — YOLOv8m Training Protocol

> **Research-grade small-object detection pipeline for soil microplastic identification**

## Overview

This notebook implements a reproducible training protocol for detecting microplastic particles
(fiber, fragment, film) in microscopy images of soil samples using **YOLOv8m**. The pipeline is
optimised for:

- **Small-object detection** — particles typically &lt;32×32 px against complex soil backgrounds
- **Research reproducibility** — deterministic seeding, fixed hyperparameters, full metric logging
- **Hardware constraints** — tuned for Google Colab Tesla T4 (15 GB VRAM)

## Training Configuration Summary

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| Model | YOLOv8m | Best accuracy / VRAM trade-off for T4 |
| Image size | 1024 | High resolution for small objects; fits T4 VRAM |
| Batch size | 4 | Maximum for YOLOv8m @ 1024 on T4 (15 GB) |
| Epochs | 200 | Sufficient convergence with early stopping |
| Optimizer | AdamW | Superior generalisation over SGD on small datasets |
| Early stopping | patience = 50 | Prevents overfitting while allowing LR recovery |
| AMP | Enabled | ~2× throughput with no accuracy loss |
| Seed | 42 | Deterministic reproducibility |

## Dataset

| Split | Images | Classes |
|-------|--------|---------|
| Train | 600 | fiber, fragment, film (balanced, augmented) |
| Val | 150 | fiber, fragment, film (balanced) |

## 1. Environment Setup

In [ ]:
# ==============================================================================
# 1. ENVIRONMENT SETUP
# Mount Google Drive for persistent storage and install dependencies.
# ==============================================================================

from google.colab import drive
drive.mount('/content/drive')

# Install Ultralytics YOLOv8 (pin minimum version for API consistency)
!pip install ultralytics>=8.1.0 --quiet

# Verify installation
import ultralytics
print(f"Ultralytics version : {ultralytics.__version__}")
print("Environment setup complete.")

## 2. GPU Verification & Reproducibility

Verify that a CUDA-capable GPU is available and configure deterministic training seeds.

**T4 Memory Budget (15 GB VRAM):**

| Configuration | Peak VRAM | Status |
|---------------|-----------|--------|
| YOLOv8m @ 1024, batch=4, AMP | ~12–13 GB | Safe |
| YOLOv8m @ 1024, batch=8, AMP | ~18+ GB | OOM |
| YOLOv8l @ 1024, batch=4, AMP | ~16+ GB | OOM |

Always reserve ≥1 GB headroom for CUDA kernel allocations and memory fragmentation.

In [ ]:
# ==============================================================================
# 2. GPU VERIFICATION & REPRODUCIBILITY
# Confirm GPU availability, report memory, and set deterministic seeds.
# ==============================================================================

import torch
import random
import numpy as np
import os

# ---------------------------------------------------------------------------
# GPU check
# ---------------------------------------------------------------------------
assert torch.cuda.is_available(), (
    "No GPU detected. Go to Runtime > Change runtime type > GPU."
)

gpu_name   = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"GPU           : {gpu_name}")
print(f"VRAM          : {gpu_mem_gb:.1f} GB")
print(f"PyTorch       : {torch.__version__}")
print(f"CUDA          : {torch.version.cuda}")

# T4 safety guidance
if "T4" in gpu_name:
    print("\n[INFO] Tesla T4 detected (15 GB VRAM).")
    print("[INFO] Batch size is set to 4 at imgsz=1024. Do NOT increase — OOM is likely.")

# ---------------------------------------------------------------------------
# Deterministic seed for reproducibility
# Research requirement: all stochastic processes must be seeded so that
# training runs are reproducible across identical hardware.
# ---------------------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

# Note: Full CUDA determinism also requires CUBLAS_WORKSPACE_CONFIG, but
# this can reduce performance. YOLO's built-in `deterministic=True` flag
# handles torch.use_deterministic_algorithms() during training.

print(f"\nReproducibility seed : {SEED}")
print("GPU verification complete.")

## 3. Dataset Verification

Validate dataset structure, check for missing images/labels, and confirm class distribution.

**Expected structure on Google Drive:**
```
MyDrive/mp-detect/data/yolo_augmented_balanced/
├── dataset.yaml
├── images/
│   ├── train/   (600 images)
│   └── val/     (150 images)
└── labels/
    ├── train/
    └── val/
```

In [ ]:
# ==============================================================================
# 3. DATASET VERIFICATION
# Confirm all paths exist, count images & labels, detect mismatches.
# ==============================================================================

import os
import yaml
from pathlib import Path

# ---------------------------------------------------------------------------
# Paths — edit DRIVE_PROJECT_PATH if your Drive layout differs
# ---------------------------------------------------------------------------
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/mp-detect"
DATASET_PATH       = f"{DRIVE_PROJECT_PATH}/data/yolo_augmented_balanced"
OUTPUT_PATH        = f"{DRIVE_PROJECT_PATH}/experiments/yolo"
YAML_PATH          = f"{DATASET_PATH}/dataset.yaml"

os.makedirs(OUTPUT_PATH, exist_ok=True)

# ---------------------------------------------------------------------------
# Verify dataset.yaml
# ---------------------------------------------------------------------------
assert os.path.exists(YAML_PATH), f"dataset.yaml not found at {YAML_PATH}"

with open(YAML_PATH) as f:
    ds_config = yaml.safe_load(f)

# ---------------------------------------------------------------------------
# Fix dataset.yaml 'path' for Colab
# The yaml may contain a Windows absolute path (e.g. D:\...) from local dev.
# YOLO resolves image/label dirs relative to 'path', so it MUST point to the
# actual dataset root on this machine (Colab).
# ---------------------------------------------------------------------------
if ds_config.get("path") != DATASET_PATH:
    print(f"[FIX] Updating dataset.yaml 'path':")
    print(f"       Old: {ds_config.get('path')}")
    print(f"       New: {DATASET_PATH}")
    ds_config["path"] = DATASET_PATH
    with open(YAML_PATH, "w") as f:
        yaml.dump(ds_config, f, default_flow_style=False)

print("\ndataset.yaml contents:")
print(yaml.dump(ds_config, default_flow_style=False))

# ---------------------------------------------------------------------------
# Count images and labels per split; detect mismatches
# ---------------------------------------------------------------------------
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

for split in ["train", "val"]:
    img_dir = Path(DATASET_PATH) / "images" / split
    lbl_dir = Path(DATASET_PATH) / "labels" / split

    assert img_dir.exists(), f"Missing directory: {img_dir}"
    assert lbl_dir.exists(), f"Missing directory: {lbl_dir}"

    images = {f.stem for f in img_dir.iterdir() if f.suffix.lower() in IMG_EXTS}
    labels = {f.stem for f in lbl_dir.iterdir() if f.suffix == ".txt"}

    missing_labels = images - labels
    orphan_labels  = labels - images

    print(
        f"[{split:5s}]  images: {len(images):4d}  |  labels: {len(labels):4d}  |  "
        f"missing labels: {len(missing_labels)}  |  orphan labels: {len(orphan_labels)}"
    )
    if missing_labels:
        print(f"  WARNING: {len(missing_labels)} images have no matching label file.")

print("\nDataset verification complete.")

### 3.1 Class & Bounding-Box Distribution Analysis

Inspect per-class label counts to confirm dataset balance and characterise bounding-box sizes.
Imbalanced classes bias the detector toward the majority class and degrade mAP50–95.
Small-object statistics inform augmentation and resolution choices.

In [ ]:
# ==============================================================================
# 3.1 CLASS & BOUNDING-BOX DISTRIBUTION ANALYSIS
# Parse all YOLO label files and report per-class counts and bbox sizes.
# ==============================================================================

from collections import Counter
from pathlib import Path
import numpy as np

CLASS_NAMES = {0: "fiber", 1: "film", 2: "fragment"}

for split in ["train", "val"]:
    lbl_dir = Path(DATASET_PATH) / "labels" / split
    class_counts = Counter()
    bbox_widths, bbox_heights = [], []

    for lbl_file in lbl_dir.glob("*.txt"):
        for line in lbl_file.read_text().strip().splitlines():
            parts = line.split()
            if len(parts) < 5:
                continue
            cls_id = int(float(parts[0]))
            w, h   = float(parts[3]), float(parts[4])
            class_counts[cls_id] += 1
            bbox_widths.append(w)
            bbox_heights.append(h)

    total = sum(class_counts.values())
    print(f"\n[{split.upper()}] — {total} objects total")
    for cls_id in sorted(class_counts):
        name  = CLASS_NAMES.get(cls_id, f"class_{cls_id}")
        count = class_counts[cls_id]
        pct   = 100.0 * count / total if total else 0
        print(f"  {name:12s}: {count:5d}  ({pct:5.1f}%)")

    if bbox_widths:
        w_arr = np.array(bbox_widths)
        h_arr = np.array(bbox_heights)
        # YOLO labels are normalised (0–1). Convert to pixels at imgsz=1024.
        IMGSZ_REF = 1024
        print(f"\n  Bbox size (px @ {IMGSZ_REF}):")
        print(f"    Median : {np.median(w_arr)*IMGSZ_REF:.0f} × {np.median(h_arr)*IMGSZ_REF:.0f}")
        print(f"    Mean   : {np.mean(w_arr)*IMGSZ_REF:.0f} × {np.mean(h_arr)*IMGSZ_REF:.0f}")
        small_count = np.sum((w_arr * IMGSZ_REF < 32) & (h_arr * IMGSZ_REF < 32))
        print(f"    Objects < 32×32 px : {small_count} / {len(w_arr)} "
              f"({100 * small_count / len(w_arr):.1f}%)")

print("\nClass distribution analysis complete.")

## 4. Model Configuration & Training

### Architecture: YOLOv8m (Medium)

**Why YOLOv8m — not nano, small, large, or xlarge?**

- **YOLOv8n / YOLOv8s**: insufficient representational capacity for fine-grained 3-class detection
  in cluttered soil backgrounds. Small models struggle with the subtle texture differences between
  fiber, fragment, and film particles.
- **YOLOv8l / YOLOv8x**: exceed T4 VRAM at imgsz=1024 with any practical batch size (≥4). Training
  at batch=2 would under-utilise the GPU and produce unstable gradient estimates.
- **YOLOv8m**: 25.9 M parameters — strong feature extraction within the T4 memory budget.

### Small-Object Detection Design Decisions

| Decision | Value | Research Justification |
|----------|-------|----------------------|
| **imgsz = 1024** | Higher resolution preserves spatial detail for objects &lt;32 px. Beyond 1024, T4 OOMs. |
| **mosaic = 1.0** | Combines 4 images → increases object density per batch; forces multi-scale learning (Bochkovskiy et al., 2020). |
| **copy_paste = 0.3** | Synthetically augments small objects by pasting them into new contexts. Proven for low-count / small-object datasets (Ghiasi et al., 2021). |
| **mixup = 0.1** | Mild inter-image blending for regularisation. Conservative to avoid washing out small features. |
| **AdamW** | Decoupled weight decay → better generalisation than SGD on small datasets (Loshchilov & Hutter, 2019). |
| **cos_lr** | Cosine annealing avoids abrupt LR drops; smoother convergence. |
| **label_smoothing = 0.05** | Mild smoothing to prevent overconfidence. Lower than default (0.1) because with only 3 classes, excessive smoothing hurts discrimination. |
| **close_mosaic = 30** | Disables mosaic for the last 30 epochs so the model fine-tunes on clean crops without stitching artefacts. |
| **degrees = ±15°** | Moderate rotation; extreme rotation can push small bboxes outside their annotations. Flips already cover 90°/180° orientations. |
| **scale = 0.5** | ±50% scale jitter — critical for multi-scale detection of variable-size particles. |
| **perspective ≈ 0** | Minimal perspective warp; too much destroys small-object bounding boxes. |
| **box loss = 7.5** | Elevated box weight prioritises precise localisation — small bboxes are extremely sensitive to 1–2 px shifts. |

In [ ]:
# ==============================================================================
# 4. MODEL CONFIGURATION & TRAINING
# All hyperparameters are annotated with research justifications.
# GPU Memory Safety: YOLOv8m @ 1024, batch=4, AMP → peak ~12–13 GB on T4.
# If OOM occurs: reduce BATCH_SIZE to 2 (will still converge, just slower).
# Do NOT increase imgsz beyond 1024 or batch beyond 4 on a T4.
# ==============================================================================

from ultralytics import YOLO

# ---------------------------------------------------------------------------
# Core hyperparameters (T4-safe @ imgsz=1024)
# ---------------------------------------------------------------------------
MODEL          = "yolov8m.pt"      # Medium: 25.9M params — best capacity/VRAM trade-off
IMGSZ          = 1024              # High res preserves <32px object detail
BATCH_SIZE     = 4                 # Max for YOLOv8m@1024 on T4 (peak ~12–13 GB)
EPOCHS         = 200               # Sufficient for 600 images with augmentation
PATIENCE       = 50                # Early stopping: halt if val mAP stalls 50 epochs
EXPERIMENT     = "mp_yolov8m_1024_adamw"

# ---------------------------------------------------------------------------
# Optimizer — AdamW with cosine LR annealing
# AdamW decouples weight decay from the gradient update, producing better
# generalisation than SGD on small datasets (Loshchilov & Hutter, 2019).
# Cosine annealing (cos_lr=True) avoids abrupt LR transitions.
# ---------------------------------------------------------------------------
OPTIMIZER      = "AdamW"
LR0            = 1e-3              # Initial LR; AdamW works well at 1e-3
LRF            = 0.01              # Final LR factor → lr_final = 1e-3 × 0.01 = 1e-5
WEIGHT_DECAY   = 5e-4              # L2 regularisation to prevent overfitting
WARMUP_EPOCHS  = 5                 # Gradual warmup stabilises early training
WARMUP_MOM     = 0.8               # Warmup momentum ramp
WARMUP_BIAS_LR = 0.1               # Separate warmup for bias parameters

# ---------------------------------------------------------------------------
# Augmentation — tuned for small-object detection on noisy soil backgrounds
# ---------------------------------------------------------------------------
# Mosaic (Bochkovskiy et al. 2020): merges 4 images into one, increasing
# effective object density and teaching multi-scale awareness.
MOSAIC         = 1.0

# Copy-paste (Ghiasi et al. 2021): pastes objects into new backgrounds.
# Highly effective for small / rare objects in cluttered scenes.
COPY_PASTE     = 0.3

# Mixup: blends two images and their labels. Conservative (0.1) to avoid
# washing out faint small-object features against soil texture.
MIXUP          = 0.1

# HSV colour jitter — soil samples exhibit variable lighting and staining.
HSV_H          = 0.015             # Hue ±1.5%
HSV_S          = 0.7               # Saturation ±70%
HSV_V          = 0.4               # Value / brightness ±40%

# Geometric augmentation — moderate to avoid corrupting small bboxes.
# Extreme rotation/shear can push small objects outside their annotations.
DEGREES        = 15                # Rotation ±15° (flips cover larger angles)
TRANSLATE      = 0.15              # Translation ±15% of image dimension
SCALE          = 0.5               # Scale jitter ±50% — critical for multi-scale
SHEAR          = 5.0               # Shear ±5° (mild; preserves small bbox integrity)
PERSPECTIVE    = 0.0005            # Minimal perspective warp (too much destroys small objects)
FLIPUD         = 0.5               # Vertical flip (microscopy has no canonical "up")
FLIPLR         = 0.5               # Horizontal flip

# Close mosaic: disable mosaic for the last N epochs so the model
# fine-tunes on clean images without stitching artefacts.
CLOSE_MOSAIC   = 30

# Label smoothing: mild (0.05) because 3-class problem — stronger
# smoothing degrades already-limited class discrimination.
LABEL_SMOOTH   = 0.05

# ---------------------------------------------------------------------------
# Loss weights
# Elevated box weight (7.5) forces the model to prioritise precise
# localisation — small bboxes are extremely sensitive to pixel-level
# regression errors, which directly impacts mAP50–95.
# ---------------------------------------------------------------------------
BOX_LOSS       = 7.5               # Box regression loss weight
CLS_LOSS       = 0.5               # Classification loss weight
DFL_LOSS       = 1.5               # Distribution focal loss weight

# ---------------------------------------------------------------------------
# Load pre-trained model
# ---------------------------------------------------------------------------
model = YOLO(MODEL)

print(f"{'='*70}")
print(f"  YOLOv8m Training — Soil Microplastic Detection")
print(f"{'='*70}")
print(f"  Model        : {MODEL}")
print(f"  Image size   : {IMGSZ}")
print(f"  Batch size   : {BATCH_SIZE}")
print(f"  Epochs       : {EPOCHS} (early stopping patience={PATIENCE})")
print(f"  Optimizer    : {OPTIMIZER}, lr0={LR0}, lrf={LRF}, cos_lr=True")
print(f"  Augmentation : mosaic={MOSAIC}, copy_paste={COPY_PASTE}, mixup={MIXUP}")
print(f"  AMP          : Enabled (mixed precision)")
print(f"  Seed         : {SEED}")
print(f"  Output       : {OUTPUT_PATH}/{EXPERIMENT}")
print(f"{'='*70}")

# ---------------------------------------------------------------------------
# TRAIN
# ---------------------------------------------------------------------------
results = model.train(
    # ----- Dataset -----
    data=YAML_PATH,

    # ----- Core training -----
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMGSZ,
    device=0,
    workers=2,                      # Colab has 2 vCPUs; more causes I/O contention
    seed=SEED,
    deterministic=True,             # Reproducible CUDA ops (slight speed cost)

    # ----- Optimizer & LR schedule -----
    optimizer=OPTIMIZER,
    lr0=LR0,
    lrf=LRF,
    momentum=0.937,                 # β1 for AdamW (YOLO default; matches literature)
    weight_decay=WEIGHT_DECAY,
    warmup_epochs=WARMUP_EPOCHS,
    warmup_momentum=WARMUP_MOM,
    warmup_bias_lr=WARMUP_BIAS_LR,
    cos_lr=True,                    # Cosine annealing schedule

    # ----- Early stopping -----
    patience=PATIENCE,              # Stop if val mAP does not improve for 50 epochs

    # ----- Data augmentation -----
    augment=True,
    hsv_h=HSV_H,
    hsv_s=HSV_S,
    hsv_v=HSV_V,
    degrees=DEGREES,
    translate=TRANSLATE,
    scale=SCALE,
    shear=SHEAR,
    perspective=PERSPECTIVE,
    flipud=FLIPUD,
    fliplr=FLIPLR,
    mosaic=MOSAIC,
    mixup=MIXUP,
    copy_paste=COPY_PASTE,
    close_mosaic=CLOSE_MOSAIC,

    # ----- Loss weights -----
    box=BOX_LOSS,
    cls=CLS_LOSS,
    dfl=DFL_LOSS,

    # ----- Regularisation -----
    label_smoothing=LABEL_SMOOTH,

    # ----- Mixed precision (AMP) -----
    # Essential on T4: halves memory for activations, ~2× throughput,
    # negligible accuracy impact on detection tasks.
    amp=True,

    # ----- Training behaviour -----
    rect=False,                     # Disabled: ensures uniform augmentation pipeline
    cache=False,                    # Colab RAM (~12 GB) cannot cache 600 imgs @ 1024
    multi_scale=False,              # Disabled: T4 cannot afford memory overhead @ 1024

    # ----- Saving -----
    project=OUTPUT_PATH,
    name=EXPERIMENT,
    exist_ok=True,
    save=True,
    save_period=25,                 # Checkpoint every 25 epochs (Drive persistence)

    # ----- Logging -----
    verbose=True,
    plots=True,
)

print(f"\n{'='*70}")
print("Training complete.")
print(f"Best weights : {OUTPUT_PATH}/{EXPERIMENT}/weights/best.pt")
print(f"Last weights : {OUTPUT_PATH}/{EXPERIMENT}/weights/last.pt")
print(f"{'='*70}")

## 5. Validation & Metrics

Post-training validation on the held-out validation split (150 images).

**Metrics reported:**
- **mAP@0.50** — mean Average Precision at IoU = 0.50 (standard detection metric)
- **mAP@0.50:0.95** — mean AP averaged over IoU 0.50–0.95 in 0.05 steps (stricter; penalises poor localisation — the primary metric for small-object quality)
- **Precision / Recall** — per-class and aggregate

In [ ]:
# ==============================================================================
# 5. VALIDATION & METRICS
# Run full validation and report research-grade metrics.
# ==============================================================================

from ultralytics import YOLO

BEST_WEIGHTS = f"{OUTPUT_PATH}/{EXPERIMENT}/weights/best.pt"
model = YOLO(BEST_WEIGHTS)

# Validate at training resolution for consistency
metrics = model.val(
    data=YAML_PATH,
    imgsz=IMGSZ,
    batch=BATCH_SIZE,
    conf=0.001,             # Low confidence for complete Precision-Recall curve
    iou=0.6,                # NMS IoU threshold
    max_det=1000,           # Max detections per image
    plots=True,             # Generate PR curves, confusion matrix, F1 curve
    save_json=True,         # COCO-format results for external evaluation tools
)

# ---------------------------------------------------------------------------
# Aggregate metrics
# ---------------------------------------------------------------------------
print(f"\n{'='*70}")
print("  VALIDATION RESULTS")
print(f"{'='*70}")
print(f"  mAP@0.50        : {metrics.box.map50:.4f}")
print(f"  mAP@0.50:0.95   : {metrics.box.map:.4f}")
print(f"  Precision (mean) : {metrics.box.mp:.4f}")
print(f"  Recall (mean)    : {metrics.box.mr:.4f}")
print(f"{'='*70}")

# ---------------------------------------------------------------------------
# Per-class metrics
# ---------------------------------------------------------------------------
CLASS_NAMES = ["fiber", "film", "fragment"]

print(f"\n  {'Class':12s}  {'AP@0.50':>8s}")
print(f"  {'-'*12}  {'-'*8}")
for i, name in enumerate(CLASS_NAMES):
    if i < len(metrics.box.ap50):
        print(f"  {name:12s}  {metrics.box.ap50[i]:8.4f}")

print(f"\nValidation plots saved to: {OUTPUT_PATH}/{EXPERIMENT}/")
print("Validation complete.")

### 5.1 Training Curves & Sample Predictions

Visualise loss curves, mAP progression, and a grid of validation-set detections for qualitative inspection.

In [ ]:
# ==============================================================================
# 5.1 TRAINING CURVES & SAMPLE PREDICTIONS
# ==============================================================================

import matplotlib.pyplot as plt
import cv2
from pathlib import Path
from IPython.display import Image, display

results_dir = Path(OUTPUT_PATH) / EXPERIMENT

# ---------------------------------------------------------------------------
# Display training plots generated by YOLO
# ---------------------------------------------------------------------------
plot_files = [
    "results.png",
    "confusion_matrix_normalized.png",
    "PR_curve.png",
    "F1_curve.png",
]

for pf in plot_files:
    plot_path = results_dir / pf
    if plot_path.exists():
        print(f"\n--- {pf} ---")
        display(Image(filename=str(plot_path), width=800))
    else:
        print(f"[SKIP] {pf} not found.")

# ---------------------------------------------------------------------------
# Sample predictions on validation images
# ---------------------------------------------------------------------------
val_images_dir = Path(DATASET_PATH) / "images" / "val"
sample_images  = sorted(val_images_dir.glob("*"))[:6]

if sample_images:
    model = YOLO(BEST_WEIGHTS)
    preds = model(
        [str(p) for p in sample_images],
        imgsz=IMGSZ,
        conf=0.25,
        iou=0.45,
        max_det=500,
    )

    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    for ax, result in zip(axes.flatten(), preds):
        img = cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(f"{len(result.boxes)} detections", fontsize=11)
        ax.axis("off")

    plt.suptitle("Sample Validation Predictions (conf ≥ 0.25)", fontsize=14, y=1.01)
    plt.tight_layout()
    save_path = results_dir / "sample_predictions.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {save_path}")

print("\nVisualisation complete.")

## 6. Model Export & Backup

Export the best weights and confirm persistent storage on Google Drive.

**Exported formats:**
- **PyTorch** (`.pt`) — native Ultralytics format for further training or Python inference
- **ONNX** (`.onnx`) — portable format for deployment (TensorRT, OpenVINO, ONNX Runtime)

In [ ]:
# ==============================================================================
# 6. MODEL EXPORT & BACKUP
# ==============================================================================

from ultralytics import YOLO
from pathlib import Path

model = YOLO(BEST_WEIGHTS)

# ---------------------------------------------------------------------------
# Export to ONNX (portable inference format)
# ---------------------------------------------------------------------------
onnx_path = model.export(format="onnx", imgsz=IMGSZ, simplify=True)
print(f"ONNX exported : {onnx_path}")

# ---------------------------------------------------------------------------
# Verify Drive backup
# ---------------------------------------------------------------------------
weights_dir = Path(OUTPUT_PATH) / EXPERIMENT / "weights"
for w in sorted(weights_dir.glob("*.pt")):
    size_mb = w.stat().st_size / 1e6
    print(f"Saved on Drive : {w}  ({size_mb:.1f} MB)")

print(f"\nAll weights persisted on Google Drive at:")
print(f"  {weights_dir}")
print("\nTo use this model locally:")
print(f'  model = YOLO("path/to/best.pt")')
print(f'  results = model("image.png", imgsz={IMGSZ})')
print("\nExport & backup complete.")

## Appendix: Resume Training After Disconnection

If the Colab session disconnects mid-training, uncomment and run the cell below to remount Drive and resume from the last checkpoint. YOLO's `resume=True` restores the optimizer state, epoch counter, and scheduler.

In [ ]:
# ==============================================================================
# APPENDIX: RESUME TRAINING AFTER DISCONNECTION
# Uncomment all lines below and run this cell ONLY if the session was
# interrupted mid-training. This restores the full training state.
# ==============================================================================

# from google.colab import drive
# drive.mount('/content/drive')

# !pip install ultralytics>=8.1.0 --quiet

# from ultralytics import YOLO

# DRIVE_PROJECT_PATH = "/content/drive/MyDrive/mp-detect"
# OUTPUT_PATH        = f"{DRIVE_PROJECT_PATH}/experiments/yolo"
# EXPERIMENT         = "mp_yolov8m_1024_adamw"
# LAST_WEIGHTS       = f"{OUTPUT_PATH}/{EXPERIMENT}/weights/last.pt"

# model = YOLO(LAST_WEIGHTS)
# results = model.train(resume=True)

# print("Resumed training complete.")

## Summary

### Complete Hyperparameter Reference

| Category | Parameter | Value | Justification |
|----------|-----------|-------|---------------|
| Architecture | model | YOLOv8m (25.9 M params) | Best capacity / VRAM trade-off for T4 15 GB |
| Resolution | imgsz | 1024 | Preserves small-object (&lt;32 px) spatial detail |
| Batch | batch | 4 | Maximum for YOLOv8m @ 1024 without OOM on T4 |
| Epochs | epochs | 200 | Sufficient convergence; early stopping prevents waste |
| Stopping | patience | 50 | Allows recovery from LR dips; halts on true plateau |
| Optimizer | AdamW | lr₀ = 1e-3, lrf = 0.01 | Decoupled weight decay; superior on small datasets |
| Schedule | cos_lr | True | Smooth annealing; avoids abrupt LR transitions |
| Warmup | warmup_epochs | 5 | Stabilises early gradients before full LR |
| Augmentation | mosaic | 1.0 | Increases effective object density per training image |
| Augmentation | copy_paste | 0.3 | Augments small / rare objects in new contexts |
| Augmentation | mixup | 0.1 | Mild regularisation; higher values degrade small-object AP |
| Augmentation | degrees | ±15° | Moderate rotation; flips cover larger angles |
| Augmentation | scale | ±50% | Critical for multi-scale detection of variable-size particles |
| Augmentation | close_mosaic | 30 | Clean fine-tuning without stitching artefacts |
| Colour | hsv_h / s / v | 0.015 / 0.7 / 0.4 | Matches variable soil lighting and staining |
| Regularisation | label_smoothing | 0.05 | Mild; 3-class problem needs sharp class boundaries |
| Loss | box / cls / dfl | 7.5 / 0.5 / 1.5 | Elevated box weight for precise small-bbox localisation |
| Precision | AMP | True | 2× throughput; essential on T4; no accuracy penalty |
| Reproducibility | seed | 42 | Deterministic training for publication-grade results |

### Small-Object Detection Considerations

1. **Resolution is the dominant factor**: at imgsz=640, a 20 px particle occupies ~0.1% of the image. At 1024, the feature pyramid retains finer spatial information, directly improving IoU for small bboxes.

2. **Mosaic + copy-paste** effectively multiply the number of small objects the model sees per epoch, addressing the severe foreground / background imbalance inherent to small-object datasets in cluttered soil imagery.

3. **Box loss weighting** (7.5) forces the model to prioritise precise localisation — for bounding boxes under 32×32 px, even a 2 px regression error can drop IoU from 0.80 to 0.55, directly impacting mAP50–95.

4. **Conservative geometric augmentation**: extreme rotation and shear can push small objects partially outside their bounding boxes, creating noisy training labels. Moderate values (±15° rotation, ±5° shear) combined with random flips provide sufficient orientation coverage.

5. **No multi-scale training on T4**: while multi-scale is beneficial in theory, the dynamic memory overhead is unpredictable at imgsz=1024 and risks OOM on 15 GB VRAM. The scale jitter augmentation (±50%) partially compensates.

6. **No image caching**: Colab provides ~12 GB system RAM, which is insufficient to cache 600 images at 1024×1024 resolution (~1.8 GB uncompressed). Disk I/O from Drive is acceptable given the small dataset size.

### Reproducibility Checklist

- [x] Seed: 42 (Python `random`, NumPy, PyTorch, CUDA)
- [x] `deterministic=True` in YOLO training config
- [x] All hyperparameters explicitly declared (no implicit defaults)
- [x] Weights checkpointed to Google Drive every 25 epochs
- [x] Ultralytics version pinned (≥8.1.0)
- [x] COCO-format JSON results saved for external evaluation